# Tool Use Pattern Hands-On Code Example

The following implementation will demonstrate this principle by first defining a simple function to simulate an information retrieval tool. Following this, an agent will be constructed and configured to leverage this tool in response to user input. The execution of this example requires the installation of the core LangChain libraries and a model-specific provider package.

In [1]:
# !pip install langchain langchain-community langchain-google-genai langgraph

> Note: Create a `.env` file in the same directory with your Google Generative AI API key:
> ```
> GOOGLE_API_KEY="<your_google_api_key_here>"
> ```

In [2]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool as langchain_tool
from langchain.agents import create_tool_calling_agent, AgentExecutor

In [3]:
load_dotenv(override=True)

True

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [5]:
@langchain_tool
def search_information(query: str) -> str:
    """Provides factual information on a given topic. Use this tool to find answers to phrases like 'capital of France' or 'weather in London?'."""
    print(f"\n--- Tool called: search_information with query: '{query}' ---")
    # Simulate a search tool with a dictionary of predefined results.
    simulated_results = {
        "capital of france": "The capital of France is Paris.",
        "weather in london": "The weather in London is currently cloudy with a chance of rain.",
        "largest planet": "The largest planet in our solar system is Jupiter.",
        "python programming": "Python is a high-level, interpreted programming language known for its readability and versatility.",
        "default": f"Simulated search result for '{query}': No specific information found, but the topic seems interesting."
    }
    result = simulated_results.get(query.lower(), simulated_results["default"])
    print(f"--- Tool result: {result} ---\n")
    return result

In [6]:
tools = [search_information]

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)

agent_executor = AgentExecutor(agent=agent, verbose=True, tools=tools)

In [7]:
async def run_agent_with_tool(query: str):
    """Invokes the agent executor with a query and prints the final response."""
    print(f"\n--- Running Agent with Query: '{query}' ---")
    try:
        response = await agent_executor.ainvoke({"input": query})
        print("\n--- Final Agent Response ---")
        print(response['output'])
    except Exception as e:
        print(f"An Error occurred during agent execution: {e}")

In [8]:
import asyncio


tasks = [
    run_agent_with_tool("What is the capital of France?"),
    run_agent_with_tool("What's the weather like in London?"),
    run_agent_with_tool("Tell me something about the dogs."),
]

await asyncio.gather(*tasks)


--- Running Agent with Query: 'What is the capital of France?' ---

--- Running Agent with Query: 'What's the weather like in London?' ---

--- Running Agent with Query: 'Tell me something about the dogs.' ---


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...

Invoking: `search_information` with `{'query': 'dogs'}`



--- Tool called: search_information with query: 'dogs' ---
--- Tool result: Simulated search result for 'dogs': No specific information found, but the topic seems interesting. ---

Simulated search result for 'dogs': No specific information found, but the topic seems interesting.
Invoking: `search_information` with `{'query': 'weather in London'}`



--- Tool called: search_information with query: 'weather in London' ---
--- Tool result: The weather in London is currently cloudy with a chance of rain. ---

The weather in London is currently cloudy with a chance of rain.
Invoking: `search_information` w

[None, None, None]